##Notebook 5: Semantic Retrieval and Evidence Engine

This notebook builds MeridianIQ's semantic evidence retrieval layer.

After detecting clauses, MeridianIQ should not simply state that a clause exists. It should also retrieve supporting evidence so that users can understand where the relevant legal language came from.

This notebook uses Sentence Transformer embeddings and cosine similarity to retrieve the most relevant clause evidence for each MVP clause. This creates the foundation for explainable contract intelligence and grounded LLM report generation.

##Loading Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json,joblib

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns",None)
pd.set_option("display.max_colwidth",300)


##Loading Clause Contexts and Risk Configuration

This step loads the clause context table and the risk configuration created in earlier notebooks.

In [ ]:
clause_contexts=pd.read_csv("clause_contexts.csv")
risk_config=pd.read_csv("clause_risk_config.csv")

clause_contexts.shape,risk_config.shape

((6702, 3), (33, 11))

##Selecting MVP Clauses

In [ ]:
mvp_config=risk_config[risk_config["is_mvp_clause"]==True].copy()
mvp_clause_names=mvp_config["clause_name"].to_list()

len(mvp_clause_names),mvp_clause_names[:5]

(23,
 ['Non-Compete',
  'Exclusivity',
  'No-Solicit Of Customers',
  'No-Solicit Of Employees',
  'Termination For Convenience'])

##Building the Evidence Corpus

The evidence corpus contains the legal clause text that MeridianIQ can search through.

Only clause contexts belonging to MVP clauses are included.

In [ ]:
evidence_corpus=clause_contexts[
    clause_contexts["clause_name"].isin(mvp_clause_names)
].copy()

evidence_corpus=evidence_corpus.dropna(subset=["context_text"]).reset_index(drop=True)
evidence_corpus["context_text"]=evidence_corpus["context_text"].astype(str)

evidence_corpus.shape


(3094, 3)

##Loading the Sentence Transformer Mode

In [ ]:
embedding_model_name="all-MiniLM-L6-v2"
embedding_model=SentenceTransformer(embedding_model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##Embedding the Evidence Corpus

Each clause evidence snippet is converted into a numerical embedding.

This transforms legal text into vectors that can be compared using cosine similarity.

In [ ]:
evidence_embeddings=embedding_model.encode(
    evidence_corpus["context_text"].to_list(),
    show_progress_bar=True,
    convert_to_numpy=True

)

evidence_embeddings.shape

Batches:   0%|          | 0/97 [00:00<?, ?it/s]

(3094, 384)

##Building Clause Queries

A retrieval system needs meaningful search queries.

Instead of searching only with short clause names such as Insurance or Non-Compete, MeridianIQ builds richer queries using both the clause name and its business description.

This improves retrieval because the search query contains more semantic context about what the clause means and why it matters.

In [ ]:
clause_queries={}

for _,row in mvp_config.iterrows():
  clause=row["clause_name"]
  description=row["business_description"]

  clause_queries[clause]= f"{clause}.{description}"

list(clause_queries.items())[:3]

[('Non-Compete',
  'Non-Compete.Restricts a party from competing in certain markets, geographies, or sectors.'),
 ('Exclusivity',
  'Exclusivity.Creates exclusive dealing obligations that may limit business flexibility.'),
 ('No-Solicit Of Customers',
  'No-Solicit Of Customers.Restricts solicitation of the counterparty’s customers or partners.')]

##Embedding Clause Queries

The clause queries are converted into embeddings using the same Sentence Transformer model.

This ensures that both the evidence snippets and the clause queries exist in the same semantic vector space

In [ ]:
query_texts=list(clause_queries.values())
query_clauses=list(clause_queries.keys())


query_embeddings=embedding_model.encode(
    query_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

query_embeddings.shape

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(23, 384)

##Computing Semantic Similarity

This step calculates cosine similarity between clause query embeddings and evidence embeddings.

Cosine similarity measures how close two pieces of text are in semantic meaning.

A higher similarity score means the evidence text is more likely to be relevant to the queried clause.

In [ ]:
similarity_matrix=cosine_similarity(query_embeddings,evidence_embeddings)
similarity_matrix.shape

(23, 3094)

##Retrieving Top Evidence Snippets

For every MVP clause query, MeridianIQ retrieves the top matching evidence snippets.

Each retrieved result includes:

Query clause
Retrieved clause
Filename
Similarity score
Evidence text
Rank

This allows MeridianIQ to not only identify relevant evidence but also rank it by semantic relevance.

In [ ]:
top_k=5
retrieval_records=[]

for query_idx,clause_name in enumerate(query_clauses):
  scores=similarity_matrix[query_idx]
  top_indices=np.argsort(scores)[::-1][:top_k]

  for rank,evidence_idx in enumerate(top_indices,start=1):
    evidence_row=evidence_corpus.iloc[evidence_idx]

    retrieval_records.append({
        "query_clause":clause_name,
        "rank":rank,
        "retrieved_clause":evidence_row["clause_name"],
        "filename":evidence_row["filename"],
        "similarity_score":scores[evidence_idx],
        "context_text":evidence_row["context_text"]
    })

retrieval_results=pd.DataFrame(retrieval_records)

retrieval_results.head(10)



,query_clause,rank,retrieved_clause,filename,similarity_score,context_text
0,Non-Compete,1,Non-Compete,IDREAMSKYTECHNOLOGYLTD_07_03_2014-EX-10.39-Cooperation Agreement on Mobile Game Business.PDF,0.535936,"['Party B shall not advertise, or make any statement favorable for, any competitor having the same or similar business scope as Party A in the services it provides.']"
1,Non-Compete,2,Non-Compete,DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement.pdf,0.515768,"['During the Term, except as otherwise provided in this Agreement, Network Affiliate and its affiliates agree not to engage or participate in any business, hold equity interests, directly or indirectly, in another entity, whether currently existing or hereafter created, or participate in any o..."
2,Non-Compete,3,Non-Compete,TUNIUCORP_03_06_2014-EX-10-COOPERATION AGREEMENT.PDF,0.507013,"[""Party A irrevocably undertakes that, without Party B's consent, Party A shall not conduct any other business or make any commercial arrangement, including without limitation being engaged in or otherwise participating in any commercial activities and businesses independently or together with a..."
3,Non-Compete,4,Exclusivity,RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT.PDF,0.501064,['No marketing exclusivity in any category or with respect to any competitors of Sponsor is conferred or implied by this Agreement except to the extent explicitly set forth in the Agreement Summary.']
4,Non-Compete,5,Non-Compete,IMMUNOMEDICSINC_08_07_2019-EX-10.1-PROMOTION AGREEMENT.PDF,0.497671,"['During the Term, neither Company nor any of its Affiliates (including, for the avoidance of doubt, any Third Party that becomes an Affiliate of Company after the Effective Date) shall, alone or in collaboration with any Third Party, market, promote, sell, distribute or otherwise commercialize ..."
5,Exclusivity,1,Anti-Assignment,"TRANSPHORM,INC_02_14_2020-EX-10.12(1)-JOINT VENTURE AGREEMENT.PDF",0.514093,"[""Notwithstanding the foregoing, no rights, obligations or liabilities hereunder shall be assignable by a Party without prior written consent of all of the other Parties; provided, however, that a Party shall not unreasonably withhold its consent to the assignment of rights and obligations by th..."
6,Exclusivity,2,Non-Compete,WPPPLC_04_30_2020-EX-4.28-SERVICE AGREEMENT.PDF,0.505805,['The Executive agrees and undertakes with the Company acting on behalf of itself and as agent for each Group Company that he will not in any Relevant Capacity at any time during the Restricted Period: (a) within or in relation to the Restricted Territory take any steps preparatory to or be dire...
7,Exclusivity,3,No-Solicit Of Customers,ENERGOUSCORP_03_16_2017-EX-10.24-STRATEGIC ALLIANCE AGREEMENT.PDF,0.500881,"['For clarity, ENERGOUS shall not intentionally supply Products, Product Die or comparable products or product die to customers directly or through distribution channels.']"
8,Exclusivity,4,Unlimited/All-You-Can-Eat-License,VERTICALNETINC_04_01_2002-EX-10.19-MAINTENANCE AND SUPPORT AGREEMENT.PDF,0.491729,"['Except as the parties may otherwise agree in writing, Converge, to the extent it has the legal right to do so, hereby grants to Vert an irrevocable, perpetual, world-wide, non-exclusive right and license to use, load, store, transmit, execute, copy, market, distribute, in any medium or distrib..."
9,Exclusivity,5,Irrevocable Or Perpetual License,VERTICALNETINC_04_01_2002-EX-10.19-MAINTENANCE AND SUPPORT AGREEMENT.PDF,0.491729,"['Except as the parties may otherwise agree in writing, Converge, to the extent it has the legal right to do so, hereby grants to Vert an irrevocable, perpetual, world-wide, non-exclusive right and license to use, load, store, transmit, execute, copy, market, distribute, in any medium or distrib..."


##Evaluating Retrieval Accuracy

Because the CUAD dataset provides known clause labels, MeridianIQ can evaluate whether the retrieved evidence matches the queried clause.

This step checks whether the retrieved clause name is the same as the query clause name.

The goal is to measure whether the semantic retrieval layer is retrieving evidence from the correct legal category.

In [ ]:
retrieval_results["is_correct_clause"]=(
    retrieval_results["query_clause"]==retrieval_results["retrieved_clause"]
)

top1_accuracy=(
    retrieval_results[retrieval_results["rank"]==1]["is_correct_clause"].mean()

)

top5_accuracy=(
    retrieval_results
    .groupby("query_clause")["is_correct_clause"]
    .any()
    .mean()
)

top1_accuracy,top5_accuracy

(np.float64(0.6956521739130435), np.float64(0.9130434782608695))

Retrieval is evaluated at different ranks.

Top-1 accuracy checks whether the best retrieved evidence is correct.

Top-5 accuracy checks whether at least one correct evidence snippet appears within the top five retrieved results.

##Per-Clause Retrieval Performance

This step summarizes retrieval quality for each clause.

Per-clause retrieval performance helps identify where MeridianIQ's evidence engine is strong and where future improvements may be needed.

In [ ]:
retrieval_summary=(
    retrieval_results
    .groupby("query_clause")
    .agg(
        top1_correct=("is_correct_clause",lambda x:bool(x.iloc[0])),
        top5_contains_correct=("is_correct_clause","any"),
        best_similarity=("similarity_score","max")
    )
    .reset_index()

)

retrieval_summary

,query_clause,top1_correct,top5_contains_correct,best_similarity
0,Anti-Assignment,True,True,0.665216
1,Audit Rights,True,True,0.651602
2,Cap On Liability,False,True,0.685749
3,Change Of Control,True,True,0.692548
4,Exclusivity,False,False,0.514093
5,Insurance,True,True,0.639853
6,Ip Ownership Assignment,False,True,0.716091
7,Irrevocable Or Perpetual License,True,True,0.617338
8,Joint Ip Ownership,True,True,0.691679
9,License Grant,True,True,0.675422


##Exporting Retrieval Outputs

The retrieval results and retrieval summary are saved for later notebooks.

In [ ]:
retrieval_summary.to_csv("clause_retrieval_summary.csv",index=False)
retrieval_results.to_csv("clause_retrieval_results.csv",index=False)
embedding_config={
    "embedding_model_name":embedding_model_name,
    "top_k":top_k
}

with open("retrieval_embedding_config.json","w") as f:
  json.dump(embedding_config,f,indent=2)

joblib.dump(evidence_embeddings,"evidence_embeddings.pkl")


['evidence_embeddings.pkl']

##Notebook Summary:

This notebook built MeridianIQ's semantic evidence retrieval engine.

The notebook produced:

1. MVP-focused evidence corpus
2. Clause query embeddings
3. Evidence text embeddings
4. Semantic similarity matrix
5. Top-K evidence retrieval results
6. Retrieval accuracy metrics
7. Per-clause retrieval summary
8. Exported retrieval artifacts

This layer is essential because MeridianIQ is not designed to be a black-box clause detector. It must also show supporting evidence for its outputs.